Sorting data into temporal datasets

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import re
import numpy as np
from datetime import datetime, timedelta
import os
import seaborn as sns

In [39]:
df = pd.read_csv("../data/unfiltered/endo_and_ibs_all_posts.csv")

In [ ]:
# Convert timestamps to datetime
df['created_date'] = pd.to_datetime(df['created_utc'], unit='s')
df['first_match_date'] = pd.to_datetime(df['first_match_time'], unit='s')

# Calculate days before first diagnosis mention
df['days_before_label'] = (df['first_match_time'] - df['created_utc']) / (24 * 3600)

print(f"Days before label range: {df['days_before_label'].min():.1f} to {df['days_before_label'].max():.1f}")

In [ ]:
#Keep only posts made MORE than 730 days before diagnosis 

TWO_YEARS_DAYS = 730
df_2yr = df[df['days_before_label'] > TWO_YEARS_DAYS].copy()

active_2yr_authors = df[df['days_before_label'] > 730]['author'].unique()
print(f"Authors active 2+ years: {len(active_2yr_authors)}")  

# Filter so only authors active 2+ years remain
df_active = df[df['author'].isin(active_2yr_authors)].copy()

In [42]:
time_windows = {
    "active_authors_exclude_1month":  30,
    "active_authors_exclude_2months": 60,
    "active_authors_exclude_3months": 90,
    "active_authors_exclude_4months": 120,
    "active_authors_exclude_5months": 150,
    "active_authors_exclude_6months": 180,
    "active_authors_exclude_7months": 210,
    "active_authors_exclude_8months": 240,
    "active_authors_exclude_9months": 270,
    "active_authors_exclude_10months": 300,
    "active_authors_exclude_11months": 330,
    "active_authors_exclude_1year":   365,
    "active_authors_exclude_13months": 395,
    "active_authors_exclude_14months": 425,
    "active_authors_exclude_15months": 455, 
    "active_authors_exclude_16months": 485,
    "active_authors_exclude_17months": 515,
    "active_authors_exclude_18months": 545,
    "active_authors_exclude_19months": 575,
    "active_authors_exclude_20months": 605,
    "active_authors_exclude_21months": 635,
    "active_authors_exclude_22months": 665,
    "active_authors_exclude_23months": 695,
    "active_authors_exclude_2years": 730
}

def create_author_documents(df_authors, min_days, window_name):
    window_data = df_authors[df_authors['days_before_label'] > min_days].copy()
    window_data = window_data.sort_values(['author', 'created_date'])

    author_docs = []
    for author, group in window_data.groupby('author'):
        first_row = group.iloc[0]
        posts = group['title'].fillna('') + ' ' + group['selftext'].fillna('')
        concatenated_text = ' '.join(posts.astype(str))

        author_docs.append({
            'author':                author,
            'label':                 first_row['label'],
            'post_count':            len(group),
            'subreddits':            ', '.join(group['subreddit'].fillna('unknown').astype(str).unique()),
            'subreddit_count':       group['subreddit'].nunique(),
            'concatenated_text':     concatenated_text,
            'text_length':           len(concatenated_text),
            'time_window':           window_name,
            'min_days_before_label': min_days,
            'time_span_days':        (group['created_date'].max() - group['created_date'].min()).days,
            'first_post_date':       group['created_date'].min(),
            'last_post_date':        group['created_date'].max(),
        })

    return pd.DataFrame(author_docs)

In [ ]:
os.makedirs('../data/unfiltered/active_authors_time_windows', exist_ok=True)

summaries = []
for window_name, min_days in time_windows.items():
    author_docs_df = create_author_documents(df_active, min_days, window_name)

    filepath = f'../data/unfiltered/active_authors_time_windows/{window_name}.csv'
    author_docs_df.to_csv(filepath, index=False)

    label_counts = author_docs_df['label'].value_counts().sort_index()
    summaries.append({
        'time_window':    window_name,
        'min_days':       min_days,
        'average_posts': author_docs_df['post_count'].mean(),
        'total_posts':    author_docs_df['post_count'].sum(),
        'endo_authors':   label_counts.get(1, 0),
        'ibs_authors':    label_counts.get(2, 0),
    })

summary_df = pd.DataFrame(summaries)
print(summary_df.to_string(index=False))